In [1]:
# =========================================================
# STEP 1 : Import Required Libraries
# =========================================================

from bs4 import BeautifulSoup
import pandas as pd

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# =========================================================
# STEP 2 : Read XML File
# =========================================================

file_path = "/content/datasets.xml"

with open(file_path, "r", encoding="ISO-8859-1") as file:
    xml_data = file.read()

print("XML File Loaded Successfully")
print("\nFirst 500 Characters:\n")
print(xml_data[:500])

XML File Loaded Successfully

First 500 Characters:

<?xml version="1.0" encoding="ISO-8859-1" ?>
<erddapDatasets>
<!-- GENERAL INFORMATION
The information in this file specifies which datasets your ERDDAP will serve.
The initial version of this file has a few examples which should work for you.
But after your initial experiments with ERDDAP, 
you should replace them with information for your datasets.
You can change this document (e.g., add datasets, change metadata) while ERDDAP is running. 
The changes will be detected the next time this docume


In [3]:
# =========================================================
# STEP 3 : Parse XML using BeautifulSoup
# =========================================================

soup = BeautifulSoup(xml_data, "xml")

print("XML Parsed Successfully")

# Show Root Tag
print("\nRoot Tag:")
print(soup.find().name)

XML Parsed Successfully

Root Tag:
erddapDatasets


In [4]:
# =========================================================
# STEP 4 : Extract Dataset Information
# =========================================================

datasets = soup.find_all("dataset")

dataset_records = []

for ds in datasets:

    dataset_info = {
        "datasetID": ds.get("datasetID"),
        "type": ds.get("type"),
        "active": ds.get("active"),
        "reloadEveryNMinutes": ds.find("reloadEveryNMinutes").text if ds.find("reloadEveryNMinutes") else None,
        "fileDir": ds.find("fileDir").text if ds.find("fileDir") else None,
        "fileNameRegex": ds.find("fileNameRegex").text if ds.find("fileNameRegex") else None,
        "sortedColumnSourceName": ds.find("sortedColumnSourceName").text if ds.find("sortedColumnSourceName") else None
    }

    dataset_records.append(dataset_info)

# Convert to DataFrame
dataset_df = pd.DataFrame(dataset_records)

print("Dataset DataFrame Created")
print("\nDataset DataFrame:\n")

print(dataset_df)

Dataset DataFrame Created

Dataset DataFrame:

             datasetID                 type active reloadEveryNMinutes  \
0  test_2788_304d_47c8  EDDTableFromNcFiles   true                   5   

                           fileDir fileNameRegex sortedColumnSourceName  
0  /Users/lcampbell/data/ooi/test/        .*\.nc     sci_m_present_time  


In [5]:
# =========================================================
# STEP 5 : Extract Data Variables
# =========================================================

variable_records = []

for ds in datasets:

    dataset_id = ds.get("datasetID")

    variables = ds.find_all("dataVariable")

    for var in variables:

        record = {
            "datasetID": dataset_id,
            "sourceName": var.find("sourceName").text if var.find("sourceName") else None,
            "destinationName": var.find("destinationName").text if var.find("destinationName") else None,
            "dataType": var.find("dataType").text if var.find("dataType") else None
        }

        variable_records.append(record)

# Convert to DataFrame
variables_df = pd.DataFrame(variable_records)

print("Variables DataFrame Created")
print("\nVariables DataFrame:\n")

print(variables_df.head(10))

Variables DataFrame Created

Variables DataFrame:

             datasetID                   sourceName  \
0  test_2788_304d_47c8                          obs   
1  test_2788_304d_47c8               sci_water_cond   
2  test_2788_304d_47c8           sci_m_present_time   
3  test_2788_304d_47c8            sci_water_pracsal   
4  test_2788_304d_47c8             driver_timestamp   
5  test_2788_304d_47c8                           id   
6  test_2788_304d_47c8                   provenance   
7  test_2788_304d_47c8                          lon   
8  test_2788_304d_47c8           internal_timestamp   
9  test_2788_304d_47c8  m_present_secs_into_mission   

               destinationName dataType  
0                          obs     long  
1               sci_water_cond   double  
2           sci_m_present_time   double  
3            sci_water_pracsal   double  
4             driver_timestamp   double  
5                           id   String  
6                   provenance   String  
7      

In [6]:
# =========================================================
# STEP 6 : DataFrame Information
# =========================================================

print("Dataset DataFrame Info")
print(dataset_df.info())

print("\nVariables DataFrame Info")
print(variables_df.info())

Dataset DataFrame Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   datasetID               1 non-null      object
 1   type                    1 non-null      object
 2   active                  1 non-null      object
 3   reloadEveryNMinutes     1 non-null      object
 4   fileDir                 1 non-null      object
 5   fileNameRegex           1 non-null      object
 6   sortedColumnSourceName  1 non-null      object
dtypes: object(7)
memory usage: 188.0+ bytes
None

Variables DataFrame Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   datasetID        23 non-null     object
 1   sourceName       23 non-null     object
 2   destinationName  23 non-null     object
 3   d

In [7]:
# =========================================================
# STEP 7 : Generate CSV Outputs
# =========================================================

dataset_csv = "dataset_information.csv"
variables_csv = "variables_information.csv"

dataset_df.to_csv(dataset_csv, index=False)

variables_df.to_csv(variables_csv, index=False)

print("CSV Files Generated Successfully")

print("\nGenerated Files:")
print(dataset_csv)
print(variables_csv)

CSV Files Generated Successfully

Generated Files:
dataset_information.csv
variables_information.csv


In [8]:
# =========================================================
# STEP 8 : Validate CSV Output
# =========================================================

dataset_validate = pd.read_csv(dataset_csv)
variables_validate = pd.read_csv(variables_csv)

print("Dataset CSV Validation")
print(dataset_validate.head())

print("\nVariables CSV Validation")
print(variables_validate.head())

Dataset CSV Validation
             datasetID                 type  active  reloadEveryNMinutes  \
0  test_2788_304d_47c8  EDDTableFromNcFiles    True                    5   

                           fileDir fileNameRegex sortedColumnSourceName  
0  /Users/lcampbell/data/ooi/test/        .*\.nc     sci_m_present_time  

Variables CSV Validation
             datasetID          sourceName     destinationName dataType
0  test_2788_304d_47c8                 obs                 obs     long
1  test_2788_304d_47c8      sci_water_cond      sci_water_cond   double
2  test_2788_304d_47c8  sci_m_present_time  sci_m_present_time   double
3  test_2788_304d_47c8   sci_water_pracsal   sci_water_pracsal   double
4  test_2788_304d_47c8    driver_timestamp    driver_timestamp   double
